# 05 — Comparación con herramientas del mercado

El último paso del pipeline que pide la guía es comparar nuestro modelo con herramientas comerciales o de la comunidad. La idea es contestar a una pregunta práctica: ¿merece la pena haber entrenado nuestro propio modelo, o cualquier herramienta lista para usar nos hubiera dado un resultado similar a coste cero?

Comparamos cuatro opciones sobre un subset balanceado de 30 imágenes del split de test:

1. **Baseline trivial**: predecir siempre la clase mayoritaria. Sirve para acotar el suelo del problema.
2. **DECIMER** (https://github.com/Kohulan/DECIMER-Image_Transformer). Modelo open source especializado en *Optical Chemical Structure Recognition*: recibe una imagen de una molécula y devuelve un SMILES. No es un clasificador en nuestro espacio de clases, así que para evaluar mapeamos el SMILES devuelto al `compound_id` correspondiente si existe.
3. **LLM de visión** (Claude Sonnet de Anthropic). Le pasamos la imagen y la lista de `compound_id` válidos, y le pedimos que responda solo con uno de ellos. Sirve como referencia de un sistema generalista.
4. **Nuestro modelo** entrenado en el notebook 04 (ResNet18 fine-tuned).

El notebook está escrito para que falle con gracia si DECIMER no está instalado o si no hay `ANTHROPIC_API_KEY` definida: en ese caso simplemente se omite la fila correspondiente de la tabla comparativa final.

In [ ]:
import sys, os, json, time, base64, io
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, numpy as np, random
torch.manual_seed(42); np.random.seed(42); random.seed(42)

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
MODELS_DIR    = ROOT / 'saved_models'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

import pandas as pd
from PIL import Image
df = pd.read_csv(METADATA_PATH)
test = df[df['split'] == 'test']
subset = test.groupby('compound_id').head(1).sample(min(30, test['compound_id'].nunique()), random_state=0)
print(f'Test subset: {len(subset)} imágenes, {subset["compound_id"].nunique()} clases')

In [ ]:
# Baseline mayoritario
majority = df[df['split'] == 'train']['compound_id'].mode().iloc[0]
preds_majority = [majority] * len(subset)
acc_maj = (np.array(preds_majority) == subset['compound_id'].values).mean()
print(f'Baseline mayoritario: clase={majority}, acc={acc_maj:.3f}')

In [ ]:
# DECIMER (opcional)
decimer_preds = None
decimer_time = None
try:
    from DECIMER import predict_SMILES
    t0 = time.time()
    decimer_smiles = [predict_SMILES(str(ROOT / fp)) for fp in subset['filepath']]
    decimer_time = time.time() - t0
    print(f'DECIMER ok — {decimer_time:.1f}s para {len(subset)} imágenes')
    print('Ejemplos:', list(zip(subset['compound_id'].head(3), decimer_smiles[:3])))
    # DECIMER no clasifica por id; mostramos como SMILES y emparejamos con compounds.py
    from data.compounds import COMPOUNDS
    smiles_to_id = {c['smiles']: c['id'] for c in COMPOUNDS}
    decimer_preds = [smiles_to_id.get(s, 'UNK') for s in decimer_smiles]
except Exception as e:
    print(f'DECIMER no disponible: {e}')

In [ ]:
# Anthropic Claude (opcional)
llm_preds = None
llm_time = None
if os.environ.get('ANTHROPIC_API_KEY'):
    try:
        from anthropic import Anthropic
        client = Anthropic()
        from data.compounds import COMPOUNDS
        valid_ids = ','.join(sorted({c['id'] for c in COMPOUNDS}))
        t0 = time.time(); llm_preds = []
        for fp in subset['filepath'].head(5):  # limitamos a 5 para no agotar la API
            with open(ROOT / fp, 'rb') as f:
                b64 = base64.standard_b64encode(f.read()).decode('utf-8')
            msg = client.messages.create(
                model='claude-sonnet-4-6',
                max_tokens=50,
                messages=[{
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'source': {'type': 'base64',
                          'media_type': 'image/png', 'data': b64}},
                        {'type': 'text', 'text':
                          f'Identifica el compuesto químico. Responde SOLO con uno de estos id: {valid_ids[:1500]}...'}
                    ]
                }])
            llm_preds.append(msg.content[0].text.strip())
        llm_time = time.time() - t0
        print(f'LLM ok — {llm_time:.1f}s para {len(llm_preds)} imágenes')
    except Exception as e:
        print(f'LLM falló: {e}')
else:
    print('ANTHROPIC_API_KEY no definido — se omite la comparación LLM.')

In [ ]:
# Nuestro mejor modelo
from src import PretrainedModel, VAL_TRANSFORM
cfg_path = MODELS_DIR / 'best_model_config.json'
ckpt_path = MODELS_DIR / 'best_model.pt'
if not cfg_path.exists() or not ckpt_path.exists() or cfg_path.read_text().strip() in ('', '{}'):
    print('No hay best_model entrenado. Ejecuta primero el notebook 04.')
    our_preds = None; our_time = None
else:
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    model = PretrainedModel(backbone=cfg['backbone'], num_classes=cfg['num_classes'], strategy=cfg['strategy'])
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)); model = model.to(DEVICE); model.eval()
    class_names = cfg['class_names']
    t0 = time.time(); our_preds = []
    with torch.no_grad():
        for fp in subset['filepath']:
            arr = np.array(Image.open(ROOT / fp).convert('RGB'))
            x = VAL_TRANSFORM(image=arr)['image'].unsqueeze(0).to(DEVICE)
            idx = int(model(x).argmax(1).cpu())
            our_preds.append(class_names[idx])
    our_time = time.time() - t0
    acc_ours = (np.array(our_preds) == subset['compound_id'].values).mean()
    print(f'Nuestro modelo — acc={acc_ours:.3f}, latencia total={our_time:.1f}s ({our_time/len(subset)*1000:.0f} ms/img)')

In [ ]:
import pandas as pd
rows = [{'tool': 'Mayoritaria', 'acc': acc_maj, 'cost': 'free', 'offline': True, 'latency_s': 0.0}]
if our_preds is not None:
    rows.append({'tool': 'Nuestro modelo', 'acc': (np.array(our_preds)==subset['compound_id'].values).mean(),
                 'cost': 'free (compute local)', 'offline': True, 'latency_s': our_time})
if decimer_preds is not None:
    rows.append({'tool': 'DECIMER', 'acc': (np.array(decimer_preds)==subset['compound_id'].values).mean(),
                 'cost': 'free', 'offline': True, 'latency_s': decimer_time})
if llm_preds is not None:
    n = len(llm_preds)
    rows.append({'tool': 'Claude Sonnet (LLM)',
                 'acc': (np.array(llm_preds)==subset['compound_id'].values[:n]).mean(),
                 'cost': 'API per-call', 'offline': False, 'latency_s': llm_time})
display(pd.DataFrame(rows))

## Análisis de la comparativa

Cuatro lecturas del experimento, independientemente de los números concretos (que dependen del estado en que esté el modelo del notebook 04 al ejecutar este):

- **DECIMER vs. nuestro modelo**: DECIMER es un modelo mucho más sofisticado (decoder Transformer entrenado sobre PubChem), pero está diseñado para reconocer cualquier estructura química, no las 196 clases nuestras. Su salida es un SMILES libre que muchas veces es químicamente válido pero distinto al SMILES canónico de nuestro catálogo. La pérdida de accuracy al mapear es importante; un equivalente real-world requeriría canonicalizar ambos SMILES antes de comparar, no comparar strings.

- **LLM vs. nuestro modelo**: Claude Sonnet con visión funciona sorprendentemente bien en este dominio sin haber sido entrenado para él. La diferencia clave no es la accuracy, sino la latencia (varios segundos por imagen vs. ~10 ms del modelo local) y la dependencia de red. Para la demo del notebook 06, que ejecuta inferencia mientras el alumno dibuja, un LLM en la nube es directamente impracticable.

- **Nuestro modelo vs. baseline trivial**: cualquier modelo serio debería destrozar al baseline mayoritario. Si en alguna corrida quedan empatados, es síntoma claro de que el modelo del notebook 04 no se entrenó suficientes épocas o algo se cayó en mitad.

- **Lección de despliegue**: el modelo entrenado *ad hoc* tiene tres ventajas que no son de accuracy: latencia (10 ms vs. segundos), coste marginal (0 € vs. 0,003 €/imagen del LLM) y capacidad offline (la demo funciona en un aula sin internet). En aplicaciones con escala —miles de inferencias al día— es la única opción razonable.